# Consultas de salud

Este notebook reúne las consultas en funciones para después llamarlas y ejecutarlas.

**Cómo usarlo en VS Code**
1. Instala las librerías (una sola vez): `pip install pandas openpyxl pyodbc ipykernel`
   y el **ODBC Driver 17 o 18 for SQL Server** de Microsoft.
2. Arriba a la derecha elige el kernel de Python.
3. Ajusta la sección **Configuración** y ejecuta las celdas en orden (o **Run All**).

**Para agregar una consulta nueva**
1. Crea una celda en la sección **Consultas** con una función que reciba la fecha y devuelva el texto del query.
2. Regístrala en el diccionario `CONSULTAS`.
3. Ejecútala con `ejecutar_consulta("NombreConsulta")`.

## 1. Librerías

In [ ]:
import datetime
from getpass import getpass
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 2. Configuración

- `USAR_SQL = False`: no se conecta a la base de datos; solo corre la prueba con datos de ejemplo.
- `USAR_SQL = True`: ejecuta las consultas contra el SQL endpoint de Microsoft Fabric.

Fabric solo acepta cuentas de Microsoft Entra ID (tu correo corporativo), no usuarios de SQL ni `Trusted_Connection`.

- Si dejas `password = ''`, la contraseña se pide con `getpass` al conectarte (no queda guardada en el notebook).
- Si tu cuenta exige MFA, cambia `authentication = 'ActiveDirectoryInteractive'`: abre la ventana de inicio de sesión de Microsoft y no usa la contraseña.

In [ ]:
USAR_SQL = False

server = 'jjwf2sltqteerjqzaniw7wlnr4-md2xqhbuaxkedfgc7z4trwzzra.datawarehouse.fabric.microsoft.com'
database = 'LH_COMFAMA_GOLD'
username = 'johangalvis@comfama.com.co'     # Agrega tu usuario
password = ''     # Agrega tu contraseña (si la dejas vacía se pide con getpass)
driver = '{ODBC Driver 17 for SQL Server}'
authentication = 'ActiveDirectoryPassword'  # Fabric exige Entra ID; 'ActiveDirectoryInteractive' si tienes MFA

DIAS_ATRAS = 3
# Para reprocesar un día puntual: FECHA_FIJA = datetime.date(2026, 9, 21)
FECHA_FIJA = None

CARPETA_SALIDA = Path.cwd() / "salidas"


def fecha_objetivo():
    """Fecha a procesar: FECHA_FIJA si está definida; si no, hoy menos DIAS_ATRAS."""
    if FECHA_FIJA is not None:
        return FECHA_FIJA
    return datetime.date.today() - datetime.timedelta(days=DIAS_ATRAS)


print("Fecha a procesar:", fecha_objetivo())

## 3. Consultas

Cada consulta es una función que recibe la fecha y devuelve el texto del query, igual que en el código original.
Los valores (fecha, códigos) se escriben directo en el SQL: así se ejecuta tal cual como en el código original, sin parámetros `?` (con parámetros el endpoint de Fabric cortó la conexión).
No hay riesgo de inyección porque esos valores los genera el propio notebook, no un usuario.

### 3.1 Resultados de laboratorio normales

Exámenes validados en la fecha, de los servicios del proyecto, cuyos componentes son **todos** `NORMAL`.
Si algún componente del mismo examen (paciente + servicio + fecha + autorización) no es NORMAL o está vacío, el examen completo se descarta.

In [ ]:
CODIGOS_RESULTADOS_NORMALES = [
    '19303', '19490', '19934', '19915', '19940', '19299', '19290', '19805', '19958',
    '19313', '197321', '19792', '19933', '191701', '19516', '19283', '19177', '19332',
    '19522', '19827', '19505', '19157', '19224', '19966', '19780', '19749', '19891',
    '19534', '19493', '19855', '19492', '19140', '19821', '19964', '194925', '19285',
    '19330', '19331', '199151',
]


def ResultadosNormales(fecha):
    codigos = ",".join(f"'{c}'" for c in CODIGOS_RESULTADOS_NORMALES)
    query = f"""
    SELECT
        t.fecha_validacion,
        t.numero_autorizacion,
        t.tipo_id_paciente,
        t.numero_id_paciente,
        t.telefono_paciente,
        t.Celular,
        t.Direccion_Electronica,
        t.codigo_servicio,
        t.descripcion_servicio,
        'NORMAL' AS interpretacion_resultado
    FROM BI_Salud.ResultadosAyudasDiagnosticas t
    WHERE CAST(t.fecha_validacion AS DATE) = '{fecha.isoformat()}'
      AND t.codigo_servicio IN ({codigos})
      AND NOT EXISTS (
            SELECT 1
            FROM BI_Salud.ResultadosAyudasDiagnosticas t2
            WHERE t2.numero_id_paciente = t.numero_id_paciente
              AND t2.codigo_servicio = t.codigo_servicio
              AND t2.fecha_validacion = t.fecha_validacion
              AND t2.numero_autorizacion = t.numero_autorizacion
              AND (t2.interpretacion_resultado <> 'NORMAL' OR t2.interpretacion_resultado IS NULL)
      )
    GROUP BY
        t.tipo_id_paciente,
        t.numero_id_paciente,
        t.telefono_paciente,
        t.Celular,
        t.Direccion_Electronica,
        t.fecha_validacion,
        t.codigo_servicio,
        t.descripcion_servicio,
        t.numero_autorizacion
    ORDER BY
        t.numero_id_paciente,
        t.fecha_validacion,
        t.codigo_servicio
    """
    return query

### 3.2 Plantilla para una consulta nueva

Copia esta celda, cambia el nombre y el SQL, y quítale los `#`.

In [ ]:
# def MiNuevaConsulta(fecha):
#     query = f"""
#     SELECT ...
#     FROM BI_Salud.MiTabla t
#     WHERE CAST(t.fecha AS DATE) = '{fecha.isoformat()}'
#     """
#     return query

### 3.3 Registro de consultas

Agrega aquí cada función nueva para poder ejecutarla por nombre.

In [ ]:
CONSULTAS = {
    "ResultadosNormales": ResultadosNormales,
    # "MiNuevaConsulta": MiNuevaConsulta,
}

list(CONSULTAS)

## 4. Funciones de ejecución

In [ ]:
_contrasena = None


def conectar():
    """Abre la conexión a Fabric con Entra ID; la contraseña se pide con getpass (solo la primera vez)."""
    import pyodbc

    global _contrasena
    if not username:
        raise ValueError("Escribe tu usuario en username (sección Configuración).")
    cadena = (
        f"DRIVER={driver};SERVER={server},1433;DATABASE={database};"
        f"Encrypt=yes;TrustServerCertificate=no;"
        f"Authentication={authentication};UID={username};"
    )
    if authentication == "ActiveDirectoryPassword":
        if _contrasena is None:
            _contrasena = password or getpass(f"Contraseña para {username}: ")
        cadena += f"PWD={{{_contrasena.replace('}', '}}')}}};"
    try:
        return pyodbc.connect(cadena, autocommit=True)
    except pyodbc.Error:
        _contrasena = None  # si falla (p. ej. contraseña errada) se vuelve a pedir
        raise


def olvidar_contrasena():
    """Borra la contraseña de memoria para que se pida de nuevo."""
    global _contrasena
    _contrasena = None


def ejecutar_consulta(nombre, fecha=None):
    """Ejecuta una consulta registrada en SQL Server y devuelve un DataFrame."""
    fecha = fecha or fecha_objetivo()
    query = CONSULTAS[nombre](fecha)
    with conectar() as conexion:
        cursor = conexion.cursor()
        cursor.execute(query)
        columnas = [c[0] for c in cursor.description]
        return pd.DataFrame.from_records(cursor.fetchall(), columns=columnas)


def guardar(df, nombre, fecha=None):
    """Guarda el resultado en salidas/<nombre>_<fecha>.xlsx (o .csv si falta openpyxl)."""
    fecha = fecha or fecha_objetivo()
    CARPETA_SALIDA.mkdir(exist_ok=True)
    ruta = CARPETA_SALIDA / f"{nombre}_{fecha.isoformat()}.xlsx"
    try:
        df.to_excel(ruta, index=False)
    except ImportError:
        ruta = ruta.with_suffix(".csv")
        df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"{nombre}: {len(df)} filas -> {ruta}")
    return ruta

### Probar la conexión (opcional)

Si algo falla, ejecuta esta celda: prueba la conexión y que la tabla responda, paso a paso.

In [ ]:
def probar_conexion(tabla="BI_Salud.ResultadosAyudasDiagnosticas"):
    pruebas = [
        ("Conexión", "SELECT 1 AS ok"),
        ("Tabla", f"SELECT TOP 5 * FROM {tabla}"),
    ]
    with conectar() as conexion:
        for nombre, sql in pruebas:
            try:
                filas = conexion.cursor().execute(sql).fetchall()
                print(f"OK   {nombre}: {len(filas)} fila(s)")
            except Exception as e:
                print(f"FALLA {nombre}: {e}")
                break


if USAR_SQL:
    probar_conexion()

## 5. Ejecución contra SQL Server

Solo corre si `USAR_SQL = True`. Ejecuta todas las consultas registradas y guarda cada resultado.

In [ ]:
resultados = {}

if USAR_SQL:
    for nombre in CONSULTAS:
        resultados[nombre] = ejecutar_consulta(nombre)
        guardar(resultados[nombre], nombre)
else:
    print("USAR_SQL = False: no se ejecutó nada contra la base de datos.")

In [ ]:
# Ver el resultado de una consulta puntual
resultados.get("ResultadosNormales", pd.DataFrame()).head(20)

## 6. Prueba sin base de datos (opcional)

Replica en pandas la lógica de `ResultadosNormales` sobre datos de ejemplo, para verificar qué entra y qué no.
También sirve con un Excel/CSV exportado de la tabla: `crudo = pd.read_excel("archivo.xlsx", dtype=str)`.

In [ ]:
LLAVE_EXAMEN = ["numero_id_paciente", "codigo_servicio", "fecha_validacion", "numero_autorizacion"]
COLUMNAS_SALIDA = [
    "fecha_validacion", "numero_autorizacion", "tipo_id_paciente", "numero_id_paciente",
    "telefono_paciente", "Celular", "Direccion_Electronica", "codigo_servicio", "descripcion_servicio",
]


def resultados_normales_pandas(df, fecha):
    df = df.copy()
    df["fecha_validacion"] = pd.to_datetime(df["fecha_validacion"])
    df["codigo_servicio"] = df["codigo_servicio"].astype(str).str.strip()

    # WHERE: fecha y códigos del proyecto
    base = df[(df["fecha_validacion"].dt.date == fecha)
              & (df["codigo_servicio"].isin(CODIGOS_RESULTADOS_NORMALES))]

    # NOT EXISTS: exámenes con algún componente no NORMAL o vacío
    interpretacion = df["interpretacion_resultado"].astype("string").str.strip().str.upper()
    no_normal = interpretacion.ne("NORMAL") | interpretacion.isna()
    examenes_malos = df.loc[no_normal, LLAVE_EXAMEN].dropna().drop_duplicates().assign(_descartar=True)
    base = base.merge(examenes_malos, on=LLAVE_EXAMEN, how="left")
    base = base[base["_descartar"].isna()]

    # GROUP BY sin agregaciones = una fila por examen
    resultado = base[COLUMNAS_SALIDA].drop_duplicates()
    resultado["interpretacion_resultado"] = "NORMAL"

    # ORDER BY
    return resultado.sort_values(
        ["numero_id_paciente", "fecha_validacion", "codigo_servicio"]
    ).reset_index(drop=True)

In [ ]:
fecha = fecha_objetivo()
hora = datetime.datetime.combine(fecha, datetime.time(9, 30))
otro_dia = hora - datetime.timedelta(days=1)

crudo = pd.DataFrame(
    [
        # 111: todos los componentes NORMAL -> SALE (1 sola fila)
        ("111", "A1", "19303", "Perfil lipídico", "NORMAL", hora),
        ("111", "A1", "19303", "Perfil lipídico", "NORMAL", hora),
        # 222: un componente ALTO -> NO sale
        ("222", "B1", "19303", "Perfil lipídico", "NORMAL", hora),
        ("222", "B1", "19303", "Perfil lipídico", "ALTO", hora),
        # 333: un componente vacío -> NO sale
        ("333", "C1", "19490", "Hemograma", "NORMAL", hora),
        ("333", "C1", "19490", "Hemograma", None, hora),
        # 444: código fuera de la lista -> NO sale
        ("444", "D1", "99999", "Otro examen", "NORMAL", hora),
        # 555: validado otro día -> NO sale
        ("555", "E1", "19934", "Glicemia", "NORMAL", otro_dia),
        # 666: sale solo el 19934; el 19915 tiene BAJO
        ("666", "F1", "19934", "Glicemia", "NORMAL", hora),
        ("666", "F2", "19915", "Creatinina", "BAJO", hora),
    ],
    columns=["numero_id_paciente", "numero_autorizacion", "codigo_servicio",
             "descripcion_servicio", "interpretacion_resultado", "fecha_validacion"],
).assign(tipo_id_paciente="CC", telefono_paciente="6041234567",
         Celular="3001234567", Direccion_Electronica="paciente@correo.com")

resultados_normales_pandas(crudo, fecha)